In [5]:
# -------------------------------------------------
# STEP 1 — Import libraries
# -------------------------------------------------

import sqlite3
import pandas as pd
from pathlib import Path


# -------------------------------------------------
# STEP 2 — Connect to NordicFlow SQLite database
# -------------------------------------------------

db_path = Path("../data/NordicFlow_ERP.db")

conn = sqlite3.connect(db_path)


# -------------------------------------------------
# STEP 3 — Load required ERP tables
# -------------------------------------------------

# Inventory transactions
inventory = pd.read_sql_query(
    "SELECT * FROM inventory_snapshot",
    conn
)

# Material planning/master data
material = pd.read_sql_query(
    "SELECT * FROM material_master",
    conn
)

print("Inventory records:", len(inventory))
print("Material records:", len(material))


# -------------------------------------------------
# STEP 4 — Prepare numeric fields
# -------------------------------------------------

# Convert inventory value to numeric
inventory["Inventory_Value_EUR"] = pd.to_numeric(
    inventory["Inventory_Value_EUR"]
        .astype(str)
        .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

# Convert unit cost to numeric
inventory["Unit_Cost_EUR"] = pd.to_numeric(
    inventory["Unit_Cost_EUR"]
        .astype(str)
        .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)


# -------------------------------------------------
# STEP 5 — Define output folder
# -------------------------------------------------

output_path = Path("../outputs")

output_path.mkdir(
    parents=True,
    exist_ok=True
)

print("Inventory notebook ready.")
# -------------------------------------------------
# STEP 8 — Prepare latest inventory snapshot
# -------------------------------------------------

# Identify the most recent inventory snapshot date
latest_date = inventory["Snapshot_Date"].max()

# Keep only the latest snapshot
latest_inventory = inventory[
    inventory["Snapshot_Date"] == latest_date
].copy()

print("Latest inventory snapshot:", latest_date)
print("Rows in latest snapshot:", len(latest_inventory))

# -------------------------------------------------
# STEP 9 — Aggregate unrestricted inventory
# -------------------------------------------------

# Keep only stock that is immediately usable
unrestricted_inventory = (
    latest_inventory[
        latest_inventory["Stock_Status"] == "Unrestricted"
    ]
    .groupby(
        ["Material_ID", "Plant_ID"],
        as_index=False
    )
    .agg(
        Unrestricted_Qty=("Quantity", "sum"),
        Inventory_Value_EUR=("Inventory_Value_EUR", "sum")
    )
)

unrestricted_inventory.head()

# -------------------------------------------------
# STEP 10 — Join inventory with material planning data
# -------------------------------------------------

inventory_risk = unrestricted_inventory.merge(
    material[
        [
            "Material_ID",
            "Material_Name",
            "Safety_Stock_Qty",
            "Reorder_Point_Qty",
            "ABC_Class",
            "XYZ_Class",
            "Criticality",
            "Preferred_Supplier_ID"
        ]
    ],
    on="Material_ID",
    how="left"
)

inventory_risk.head()

# -------------------------------------------------
# STEP 11 — Detect materials below safety stock
# -------------------------------------------------

inventory_risk["Below_Safety_Stock"] = (
    inventory_risk["Unrestricted_Qty"]
    < inventory_risk["Safety_Stock_Qty"]
)

inventory_risk["Safety_Stock_Gap"] = (
    inventory_risk["Safety_Stock_Qty"]
    - inventory_risk["Unrestricted_Qty"]
)

safety_stock_shortages = inventory_risk[
    inventory_risk["Below_Safety_Stock"]
].copy()

safety_stock_shortages[
    [
        "Plant_ID",
        "Material_ID",
        "Material_Name",
        "Criticality",
        "Unrestricted_Qty",
        "Safety_Stock_Qty",
        "Safety_Stock_Gap"
    ]
].sort_values(
    ["Criticality", "Safety_Stock_Gap"],
    ascending=[True, False]
)

# -------------------------------------------------
# STEP 12 — Detect materials below reorder point
# -------------------------------------------------

inventory_risk["Below_Reorder_Point"] = (
    inventory_risk["Unrestricted_Qty"]
    < inventory_risk["Reorder_Point_Qty"]
)

inventory_risk["Reorder_Gap"] = (
    inventory_risk["Reorder_Point_Qty"]
    - inventory_risk["Unrestricted_Qty"]
)

reorder_risk = inventory_risk[
    inventory_risk["Below_Reorder_Point"]
].copy()

reorder_risk.head(10)

# -------------------------------------------------
# STEP 13 — Detect potential C-class overstock
# -------------------------------------------------

c_class_overstock = inventory_risk[
    (inventory_risk["ABC_Class"] == "C")
    &
    (
        inventory_risk["Unrestricted_Qty"]
        > inventory_risk["Reorder_Point_Qty"]
    )
].copy()

c_class_overstock["Excess_Qty"] = (
    c_class_overstock["Unrestricted_Qty"]
    - c_class_overstock["Reorder_Point_Qty"]
)

c_class_overstock[
    [
        "Plant_ID",
        "Material_ID",
        "Material_Name",
        "Unrestricted_Qty",
        "Reorder_Point_Qty",
        "Excess_Qty",
        "Inventory_Value_EUR"
    ]
].sort_values(
    "Inventory_Value_EUR",
    ascending=False
)

# -------------------------------------------------
# STEP 14 — Identify interplant transfer opportunities
# -------------------------------------------------

transfer_candidates = inventory_risk.merge(
    inventory_risk[
        [
            "Material_ID",
            "Plant_ID",
            "Unrestricted_Qty",
            "Safety_Stock_Qty"
        ]
    ],
    on="Material_ID",
    suffixes=("_Shortage", "_Source")
)

# Keep only different plants
transfer_candidates = transfer_candidates[
    transfer_candidates["Plant_ID_Shortage"]
    != transfer_candidates["Plant_ID_Source"]
]

# Shortage plant must be below safety stock
# Source plant must be above safety stock
transfer_candidates = transfer_candidates[
    (
        transfer_candidates["Unrestricted_Qty_Shortage"]
        < transfer_candidates["Safety_Stock_Qty_Shortage"]
    )
    &
    (
        transfer_candidates["Unrestricted_Qty_Source"]
        > transfer_candidates["Safety_Stock_Qty_Source"]
    )
].copy()

# Calculate indicative transferable quantity
transfer_candidates["Indicative_Transfer_Qty"] = transfer_candidates[
    [
        "Safety_Stock_Gap",
        "Unrestricted_Qty_Source"
    ]
].min(axis=1)

transfer_candidates[
    [
        "Material_ID",
        "Material_Name",
        "Plant_ID_Shortage",
        "Unrestricted_Qty_Shortage",
        "Plant_ID_Source",
        "Unrestricted_Qty_Source"
    ]
]

# -------------------------------------------------
# STEP 15 — Calculate realistic inter-plant transfer quantity
# -------------------------------------------------

# Calculate surplus available at the source plant
transfer_candidates["Source_Surplus_Qty"] = (
    transfer_candidates["Unrestricted_Qty_Source"]
    - transfer_candidates["Safety_Stock_Qty_Source"]
)

# Calculate shortage at the receiving plant
transfer_candidates["Shortage_Qty"] = (
    transfer_candidates["Safety_Stock_Qty_Shortage"]
    - transfer_candidates["Unrestricted_Qty_Shortage"]
)

# Recommended transfer = smaller of shortage or source surplus
transfer_candidates["Indicative_Transfer_Qty"] = (
    transfer_candidates[
        ["Shortage_Qty", "Source_Surplus_Qty"]
    ]
    .min(axis=1)
)

# Keep only positive transfer opportunities
interplant_transfer = transfer_candidates[
    transfer_candidates["Indicative_Transfer_Qty"] > 0
].copy()

# Show the final transfer report
interplant_transfer[
    [
        "Material_ID",
        "Material_Name",
        "Plant_ID_Shortage",
        "Unrestricted_Qty_Shortage",
        "Safety_Stock_Qty_Shortage",
        "Plant_ID_Source",
        "Unrestricted_Qty_Source",
        "Safety_Stock_Qty_Source",
        "Indicative_Transfer_Qty"
    ]
]

# -------------------------------------------------
# STEP 16 — Export inventory analysis outputs
# -------------------------------------------------

inventory_risk.to_csv(
    output_path / "inventory_risk.csv",
    index=False
)

safety_stock_shortages.to_csv(
    output_path / "safety_stock_shortages.csv",
    index=False
)

reorder_risk.to_csv(
    output_path / "reorder_point_risk.csv",
    index=False
)

c_class_overstock.to_csv(
    output_path / "c_class_overstock.csv",
    index=False
)

interplant_transfer.to_csv(
    output_path / "interplant_transfer_opportunities.csv",
    index=False
)

print("Inventory analysis files exported successfully.")

Inventory records: 271
Material records: 20
Inventory notebook ready.
Latest inventory snapshot: 31/03/2026
Rows in latest snapshot: 90
Inventory analysis files exported successfully.
